# Generate Results

**One notebook, one Colab session, all submission results.**

Populates the entire `results/` folder:
1. `results/sample_inputs/` — sample images used
2. `results/json_outputs/` — JSON extraction for 10+ documents
3. `results/eval_summary.csv` — evaluation metrics table
4. `results/qualitative/` — 5+ side-by-side input/output images

**Model:** Qwen3-VL-2B (~4 GB VRAM, ~8s/image)  
**Runtime:** ~15 min total on Colab T4

In [ ]:
# Cell 1: Install dependencies
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q accelerate bitsandbytes qwen-vl-utils[decord] datasets pillow
!pip install -q PyMuPDF opencv-python-headless matplotlib pandas

In [ ]:
# Cell 2: Clone repo + setup
import os, sys

REPO_DIR = "VLM_PDF_Extractor"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/RahulReadd/VLM_PDF_Extractor.git
os.chdir(REPO_DIR)
sys.path.insert(0, ".")

# Create results directories
for d in ["results/sample_inputs", "results/json_outputs", "results/qualitative"]:
    os.makedirs(d, exist_ok=True)

import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB")

In [ ]:
# Cell 3: Load model
from app.extract import DocumentExtractor

MODEL_NAME = "qwen3-vl-2b"
extractor = DocumentExtractor(model_name=MODEL_NAME)
print(f"Model ready: {MODEL_NAME}")

---
## Part 1: CORD Receipt Extraction (12 documents)

In [ ]:
# Cell 4: Load CORD dataset
import json
import numpy as np
from datasets import load_dataset
from app.evaluate import (
    evaluate_single, parse_cord_ground_truth,
    evaluate_signature_single, evaluate_signature_batch,
)

cord = load_dataset("naver-clova-ix/cord-v2", split="test")

CORD_N = 12
cord_indices = np.linspace(0, len(cord) - 1, CORD_N, dtype=int).tolist()
cord_images = [cord[i]["image"].convert("RGB") for i in cord_indices]
cord_gts = [cord[i]["ground_truth"] for i in cord_indices]

print(f"Loaded {len(cord_images)} CORD receipt images")

In [ ]:
# Cell 5: Run extraction on CORD + evaluate
cord_results = []

for i, (img, gt_raw) in enumerate(zip(cord_images, cord_gts)):
    pred = extractor.extract(img)  # unified prompt
    gt_parsed = parse_cord_ground_truth(gt_raw)

    # For evaluation: unwrap receipt from unified output
    eval_json = pred["parsed_json"]
    if eval_json and "receipt" in eval_json and isinstance(eval_json["receipt"], dict):
        eval_json = eval_json["receipt"]

    scores = evaluate_single(eval_json, gt_parsed)

    result = {
        "image_idx": int(cord_indices[i]),
        "source": "cord",
        "model": MODEL_NAME,
        "prediction": pred["parsed_json"],
        "json_valid": pred["json_valid"],
        "scores": scores,
    }
    cord_results.append(result)

    # Save JSON output
    out_path = f"results/json_outputs/cord_{i:03d}.json"
    with open(out_path, "w") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    # Save sample input image
    img.save(f"results/sample_inputs/cord_{i:03d}.png")

    print(f"  [{i+1}/{CORD_N}] JSON={pred['json_valid']} | Field F1={scores['field_f1']:.3f} | Menu F1={scores['menu_f1']:.3f}")

# Aggregate CORD metrics
cord_n = len(cord_results)
cord_json_rate = sum(r["json_valid"] for r in cord_results) / cord_n
cord_field_em = sum(r["scores"]["field_em"] for r in cord_results) / cord_n
cord_field_f1 = sum(r["scores"]["field_f1"] for r in cord_results) / cord_n
cord_menu_f1 = sum(r["scores"]["menu_f1"] for r in cord_results) / cord_n

print(f"\nCORD Results ({cord_n} images):")
print(f"  JSON valid:  {cord_json_rate:.0%}")
print(f"  Field EM:    {cord_field_em:.3f}")
print(f"  Field F1:    {cord_field_f1:.3f}")
print(f"  Menu F1:     {cord_menu_f1:.3f}")

---
## Part 2: Signature Detection Evaluation (20 documents)

In [ ]:
# Cell 6: Load signature dataset (positive) + CORD as negative
SIG_N = 10  # per class → 20 total

# Positive: documents WITH signatures
sig_ds = load_dataset("Mels22/SigDetectVerifyFlow", split="test")
sig_pos_indices = np.linspace(0, len(sig_ds) - 1, SIG_N, dtype=int).tolist()
sig_pos_images = [sig_ds[i]["document"].convert("RGB") for i in sig_pos_indices]

# Negative: CORD receipts (no signatures) — use different indices than Part 1
sig_neg_indices = np.linspace(0, len(cord) - 1, SIG_N, dtype=int).tolist()
sig_neg_images = [cord[i]["image"].convert("RGB") for i in sig_neg_indices]

sig_all_images = sig_pos_images + sig_neg_images
sig_all_labels = [True] * SIG_N + [False] * SIG_N

print(f"Loaded {len(sig_all_images)} images ({SIG_N} with signatures, {SIG_N} without)")

In [ ]:
# Cell 7: Run signature detection + evaluate
sig_predictions = []
sig_results_raw = []

for i, (img, gt_label) in enumerate(zip(sig_all_images, sig_all_labels)):
    pred = extractor.extract(img)  # unified prompt
    sig_predictions.append(pred["parsed_json"])

    source = "sigdetect" if i < SIG_N else "cord"
    result = {
        "image_idx": i,
        "source": source,
        "model": MODEL_NAME,
        "gt_signature_present": gt_label,
        "prediction": pred["parsed_json"],
        "json_valid": pred["json_valid"],
    }
    sig_results_raw.append(result)

    # Save JSON
    out_path = f"results/json_outputs/sig_{i:03d}.json"
    with open(out_path, "w") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    # Save sample input
    img.save(f"results/sample_inputs/sig_{i:03d}.png")

    tag = "+" if gt_label else "-"
    print(f"  [{tag}] {i+1}/{len(sig_all_images)} ({source}): JSON={pred['json_valid']}")

# Compute metrics
sig_metrics = evaluate_signature_batch(sig_predictions, sig_all_labels)

print(f"\nSignature Detection Results ({sig_metrics['n']} images):")
print(f"  Accuracy:  {sig_metrics['accuracy']:.3f}")
print(f"  Precision: {sig_metrics['precision']:.3f}")
print(f"  Recall:    {sig_metrics['recall']:.3f}")
print(f"  F1:        {sig_metrics['f1']:.3f}")
print(f"  TP={sig_metrics['tp']}  FP={sig_metrics['fp']}  FN={sig_metrics['fn']}  TN={sig_metrics['tn']}")

---
## Part 3: Evaluation Metrics Summary (CSV)

In [ ]:
# Cell 8: Save evaluation summary CSV
import pandas as pd

summary = pd.DataFrame([
    {
        "Task": "Receipt Extraction (CORD)",
        "Dataset": "naver-clova-ix/cord-v2",
        "N": cord_n,
        "Model": MODEL_NAME,
        "JSON_Valid_Rate": round(cord_json_rate, 3),
        "Field_EM": round(cord_field_em, 3),
        "Field_F1": round(cord_field_f1, 3),
        "Menu_F1": round(cord_menu_f1, 3),
        "Accuracy": "",
        "Precision": "",
        "Recall": "",
        "Sig_F1": "",
    },
    {
        "Task": "Signature Detection",
        "Dataset": "SigDetectVerifyFlow + CORD",
        "N": sig_metrics["n"],
        "Model": MODEL_NAME,
        "JSON_Valid_Rate": "",
        "Field_EM": "",
        "Field_F1": "",
        "Menu_F1": "",
        "Accuracy": sig_metrics["accuracy"],
        "Precision": sig_metrics["precision"],
        "Recall": sig_metrics["recall"],
        "Sig_F1": sig_metrics["f1"],
    },
])

csv_path = "results/eval_summary.csv"
summary.to_csv(csv_path, index=False)
print(f"Saved evaluation summary to: {csv_path}")
print()
print(summary.to_string(index=False))

---
## Part 4: Qualitative Side-by-Side Examples (5+)

In [ ]:
# Cell 9: Generate side-by-side qualitative images
import matplotlib.pyplot as plt
import textwrap


def make_qualitative(image, pred_json, title, ground_truth_text, save_path):
    """Create a side-by-side PNG: input image | extracted JSON | ground truth."""
    fig, axes = plt.subplots(1, 3, figsize=(24, 8),
                             gridspec_kw={"width_ratios": [1, 1, 1]})

    # Left: input image
    axes[0].imshow(image)
    axes[0].set_title("Input Image", fontsize=14, fontweight="bold")
    axes[0].axis("off")

    # Middle: model prediction
    pred_text = json.dumps(pred_json, indent=2, ensure_ascii=False) if pred_json else "(no valid JSON)"
    pred_wrapped = textwrap.fill(pred_text, width=60)
    if len(pred_wrapped) > 1500:
        pred_wrapped = pred_wrapped[:1500] + "\n... [truncated]"
    axes[1].text(0.02, 0.98, pred_wrapped, transform=axes[1].transAxes,
                 fontsize=7, verticalalignment="top", fontfamily="monospace",
                 bbox=dict(boxstyle="round", facecolor="#e8f5e9", alpha=0.8))
    axes[1].set_title("Model Prediction", fontsize=14, fontweight="bold")
    axes[1].axis("off")

    # Right: ground truth
    gt_wrapped = textwrap.fill(ground_truth_text, width=60)
    if len(gt_wrapped) > 1500:
        gt_wrapped = gt_wrapped[:1500] + "\n... [truncated]"
    axes[2].text(0.02, 0.98, gt_wrapped, transform=axes[2].transAxes,
                 fontsize=7, verticalalignment="top", fontfamily="monospace",
                 bbox=dict(boxstyle="round", facecolor="#e3f2fd", alpha=0.8))
    axes[2].set_title("Ground Truth", fontsize=14, fontweight="bold")
    axes[2].axis("off")

    fig.suptitle(title, fontsize=16, fontweight="bold", y=1.02)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Saved: {save_path}")


print("Generating qualitative examples...\n")

In [ ]:
# Cell 10: Generate 3 CORD receipt examples
print("--- CORD Receipt Examples ---")
for i in range(3):
    img = cord_images[i]
    pred = cord_results[i]["prediction"]
    gt_parsed = parse_cord_ground_truth(cord_gts[i])
    gt_text = json.dumps(gt_parsed, indent=2, ensure_ascii=False)
    scores = cord_results[i]["scores"]

    title = (f"CORD Receipt #{i+1} — "
             f"Field F1: {scores['field_f1']:.3f} | Menu F1: {scores['menu_f1']:.3f}")

    make_qualitative(
        image=img,
        pred_json=pred,
        title=title,
        ground_truth_text=gt_text,
        save_path=f"results/qualitative/cord_example_{i+1}.png",
    )

In [ ]:
# Cell 11: Generate 2 signature detection examples (1 positive, 1 negative)
from app.evaluate import parse_signature_prediction

print("--- Signature Detection Examples ---")

# Example 1: Document WITH signature (positive)
pos_idx = 0
pos_img = sig_all_images[pos_idx]
pos_pred = sig_predictions[pos_idx]
pos_detected = parse_signature_prediction(pos_pred)
pos_gt = "Ground Truth: Signature PRESENT"
pos_title = f"Signature Doc (Positive) — Detected: {pos_detected} | GT: True"

make_qualitative(
    image=pos_img,
    pred_json=pos_pred,
    title=pos_title,
    ground_truth_text=pos_gt,
    save_path="results/qualitative/sig_positive_example.png",
)

# Example 2: Receipt WITHOUT signature (negative)
neg_idx = SIG_N  # first negative sample
neg_img = sig_all_images[neg_idx]
neg_pred = sig_predictions[neg_idx]
neg_detected = parse_signature_prediction(neg_pred)
neg_gt = "Ground Truth: Signature NOT PRESENT"
neg_title = f"Receipt (Negative) — Detected: {neg_detected} | GT: False"

make_qualitative(
    image=neg_img,
    pred_json=neg_pred,
    title=neg_title,
    ground_truth_text=neg_gt,
    save_path="results/qualitative/sig_negative_example.png",
)

In [ ]:
# Cell 12: Generate 1 more CORD example with full unified output visible
print("--- Full Unified Extraction Example ---")
i = 5  # pick a different CORD image
img = cord_images[i]
pred = cord_results[i]["prediction"]
gt_parsed = parse_cord_ground_truth(cord_gts[i])
gt_text = json.dumps(gt_parsed, indent=2, ensure_ascii=False)
scores = cord_results[i]["scores"]

title = (f"Unified Extraction Example — All 4 aspects in one pass\n"
         f"Field F1: {scores['field_f1']:.3f} | Menu F1: {scores['menu_f1']:.3f}")

make_qualitative(
    image=img,
    pred_json=pred,
    title=title,
    ground_truth_text=gt_text,
    save_path="results/qualitative/unified_example.png",
)

---
## Part 5: Final Summary

In [ ]:
# Cell 13: Print final summary of everything generated
from pathlib import Path

print("=" * 60)
print("  RESULTS FOLDER — SUBMISSION READY")
print("=" * 60)

json_count = len(list(Path("results/json_outputs").glob("*.json")))
sample_count = len(list(Path("results/sample_inputs").glob("*.png")))
qual_count = len(list(Path("results/qualitative").glob("*.png")))
csv_exists = Path("results/eval_summary.csv").exists()

print(f"\n  results/json_outputs/    : {json_count} JSON files (required: 10+) {'OK' if json_count >= 10 else 'NEED MORE'}")
print(f"  results/sample_inputs/   : {sample_count} sample images")
print(f"  results/qualitative/     : {qual_count} side-by-side PNGs (required: 5+) {'OK' if qual_count >= 5 else 'NEED MORE'}")
print(f"  results/eval_summary.csv : {'EXISTS' if csv_exists else 'MISSING'}")

print(f"\n  Model: {MODEL_NAME}")
print(f"  Total documents processed: {json_count}")

print(f"\n{'─'*60}")
print(f"  Receipt Extraction (CORD, n={cord_n}):")
print(f"    JSON valid: {cord_json_rate:.0%}")
print(f"    Field EM:   {cord_field_em:.3f}")
print(f"    Field F1:   {cord_field_f1:.3f}")
print(f"    Menu F1:    {cord_menu_f1:.3f}")

print(f"\n  Signature Detection (n={sig_metrics['n']}):")
print(f"    Accuracy:   {sig_metrics['accuracy']:.3f}")
print(f"    Precision:  {sig_metrics['precision']:.3f}")
print(f"    Recall:     {sig_metrics['recall']:.3f}")
print(f"    F1:         {sig_metrics['f1']:.3f}")
print(f"{'─'*60}")

# List all generated files
print(f"\n  All generated files:")
for root, dirs, files in os.walk("results"):
    for f in sorted(files):
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f"    {path} ({size:,} bytes)")

In [ ]:
# Cell 14: Commit results back to repo (optional)
# Uncomment the lines below to push results to your GitHub repo

# !git add results/
# !git commit -m "Add generated results: JSON outputs, eval summary, qualitative examples"
# !git push origin main